> **Optional advanced material.** This notebook is part of the *advanced,
> optional path* of this series — interpretability research aimed at data
> scientists and ML practitioners who want to look inside a real language
> model's internals. It goes well beyond what's needed for the main
> oilfield lessons and assumes comfort with Python, PyTorch tensors, and
> general deep-learning concepts.
>
> **You do not need this notebook** to get practical value from the main
> path — see `../notebooks/01_how_a_real_llm_predicts_the_next_token.ipynb`
> and `../notebooks/02_temperature_sampling_and_decoding_strategies.ipynb`,
> or the project [README](../README.md), for that.


# Individual-Neuron Analysis: A Suppressor Found Two Independent Ways

**Notebook 8 of the series.** Notebook 7 went from a whole layer down to
one attention head out of 12. This notebook goes one level finer still:
from a head down to one **neuron** inside a layer's feed-forward (MLP)
block — one of 8,960 per layer in this model. There are far too many
neurons to exhaustively ablate the way we exhaustively ablated 336 heads
in notebook 7, so this notebook uses a genuinely different strategy: a
cheap, approximate screening step, followed by real, expensive validation
only on the candidates the screening surfaces — and it reports honestly
where that cheap screening turns out to be *wrong*.

The notebook's centerpiece is a real finding, confirmed two independent
ways: one specific neuron, found first by gradient screening on one real
prompt, and then separately by a prompt-independent inspection of that
neuron's learned output weights alone, turns out to suppress essentially
every spelling and capitalization variant of one specific word more
strongly than any of the other 151,936 tokens in the model's vocabulary.

Still out of scope: exhaustively searching every neuron in every layer,
and testing this neuron's behavior across a large, systematic set of
prompts (which this notebook does only partially, as an exercise).

## Learning objectives

1. Why can't individual MLP neurons be exhaustively ablated the way
   attention heads were in notebook 7 — and what's a workable alternative?
2. What does "gradient x activation" screening give you cheaply, and where
   does notebook 4's "saturation" caution show up concretely, with real
   numbers, when that cheap method is checked against real ablation?
3. What is a "logit lens" projection of a neuron's output weights, and why
   doesn't it require running the model on any input at all?
4. Why is agreement between a prompt-specific causal result and a
   prompt-independent weights-only result particularly strong evidence?
5. What does — and doesn't — a clean single-neuron story like this one
   prove about the model in general?


## 1. What problem are we investigating?

Notebook 7's attention heads were countable: 12 per layer, 336 total for a
full grid, computed in about 15 seconds. A transformer's feed-forward (MLP)
block has no such small number — this model has **8,960** intermediate
dimensions per layer, often informally called "neurons." Exhaustively
ablating all of them across all 28 layers would mean 250,880 forward
passes — not something to run interactively on a laptop.

So this notebook uses a two-stage strategy instead:

1. **Cheap screening**: compute, in a single backward pass, how much each
   of the 8,960 neurons in one layer's output would need to change to
   explain the target prediction — the same "gradient x activation"
   attribution notebook 4 used on input tokens, now applied to internal
   neurons.
2. **Real validation**: take only the handful of neurons that screening
   flags as most important, and check each one with a real, individual
   ablation — the actually reliable but expensive method.

Where these two disagree is itself an important, honest finding, not
something to smooth over.


## Installation (run once)

Same dependencies as the rest of the series. If already installed, skip
this cell.


In [1]:
# Run this once. After the packages are installed you can leave this commented out.
# %pip install torch transformers accelerate pandas matplotlib numpy
print("If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.")


If an import in the next cell fails, uncomment and run the pip install line above, then re-run this notebook.


## 2. Load the real language model

Standard setup, matching notebooks 1–2, 4–6 — the default float16-on-
accelerated-hardware configuration is fine here: gradient computation
through the MLP (unlike attention weight *extraction* in notebooks 3 and 7)
does not require `eager` attention or forced `float32`.


In [2]:
import random
import sys

import numpy as np
import pandas as pd
import torch
import transformers
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PRIMARY_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # used only if the primary model fails to load

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

DTYPE = torch.float16 if DEVICE in ("mps", "cuda") else torch.float32

print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Selected device:      {DEVICE}")
print(f"Selected dtype:       {DTYPE}")


Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Selected device:      mps
Selected dtype:       torch.float16


In [3]:
def load_model(model_name: str = PRIMARY_MODEL_NAME):
    '''Load a causal language model and its tokenizer onto the selected device.

    Falls back to FALLBACK_MODEL_NAME if the primary model cannot be loaded,
    and always reports which model actually ended up running.
    '''
    try:
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModelForCausalLM.from_pretrained(model_name, dtype=DTYPE)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, model_name
    except Exception as exc:  # noqa: BLE001 - report and fall back, don't crash the notebook
        print(f"Could not load '{model_name}' ({exc}). Falling back to '{FALLBACK_MODEL_NAME}'.")
        tok = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
        mdl = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL_NAME, dtype=DTYPE)
        mdl.to(DEVICE)
        mdl.eval()
        return tok, mdl, FALLBACK_MODEL_NAME


tokenizer, model, MODEL_NAME = load_model()
NUM_LAYERS = model.config.num_hidden_layers
INTERMEDIATE_SIZE = model.config.intermediate_size
print(f"\nModel actually loaded and used in this notebook: {MODEL_NAME}")
print(f"Transformer layers: {NUM_LAYERS}, MLP intermediate size (neurons per layer): {INTERMEDIATE_SIZE:,}")
print(f"Tied input/output embeddings: {model.config.tie_word_embeddings}")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Model actually loaded and used in this notebook: Qwen/Qwen2.5-1.5B-Instruct
Transformer layers: 28, MLP intermediate size (neurons per layer): 8,960
Tied input/output embeddings: True


## 3. Reference prompt and target token

The same primary sentence and target-token convention used throughout this
series.


In [4]:
PROMPT = "The crew pulled out of hole with the worn PDC bit and prepared to run a new"
LAYER = NUM_LAYERS - 1  # the final layer -- motivated by notebooks 5 and 7 both finding late layers most causally important

encoded = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    clean_logits = model(**encoded).logits[0, -1, :]

TARGET_ID = int(torch.argmax(clean_logits).item())
TARGET_TEXT = tokenizer.decode([TARGET_ID])
clean_logprob = torch.log_softmax(clean_logits.float(), dim=-1)[TARGET_ID].item()

print(f"Prompt: {PROMPT!r}")
print(f"Target token (model's own top prediction): {TARGET_TEXT!r}")
print(f"Clean log P({TARGET_TEXT!r}) = {clean_logprob:.4f}")
print(f"Analyzing layer {LAYER} (the final transformer layer)")


Prompt: 'The crew pulled out of hole with the worn PDC bit and prepared to run a new'
Target token (model's own top prediction): ' bit'
Clean log P(' bit') = -1.0929
Analyzing layer 27 (the final transformer layer)


## 4. What is a "neuron" in this model's feed-forward block?

Each transformer layer's feed-forward block computes:

$$\text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \odot \text{up\_proj}(x)\big)$$

`gate_proj` and `up_proj` each expand the 1,536-dimensional residual stream
up to 8,960 dimensions; after the elementwise product and activation, each
of those 8,960 values is informally called a "neuron" — a single scalar
that `down_proj` then mixes back down into a 1,536-dimensional update to
the residual stream. We hook `down_proj`'s *input* — the real, computed
value of all 8,960 neurons at once, for the actual prompt above.


## 5. Cheap screening across all 8,960 neurons at once

A forward hook lets us capture the real neuron-activation tensor and mark
it to retain its gradient; one backward pass on the target log-probability
then gives us the gradient with respect to *every* neuron simultaneously —
no per-neuron forward passes needed for this step. Multiplying each
neuron's gradient by its own real activation value (exactly notebook 4's
gradient x input, applied here to neurons instead of input embeddings)
gives a cheap, approximate importance score for all 8,960 neurons in a
single pass.


In [5]:
captured = {}

def capture_hook(module, inputs, output):
    activation = inputs[0]
    activation.retain_grad()
    captured["activation"] = activation


handle = model.model.layers[LAYER].mlp.down_proj.register_forward_hook(capture_hook)
out = model(**encoded)
logprob = torch.log_softmax(out.logits[0, -1, :].float(), dim=-1)[TARGET_ID]
logprob.backward()
handle.remove()

activation = captured["activation"][0, -1, :].detach().float()
gradient = captured["activation"].grad[0, -1, :].float()
screening_scores = gradient * activation

top_scores, top_neurons = torch.topk(screening_scores.abs(), 10)
screening_table = pd.DataFrame({
    "neuron": top_neurons.tolist(),
    "screening_score": screening_scores[top_neurons].tolist(),
    "activation": activation[top_neurons].tolist(),
    "gradient": gradient[top_neurons].tolist(),
})
screening_table


,neuron,screening_score,activation,gradient
0,7015,-4.449463,31.640625,-0.140625
1,8715,0.322794,7.539062,0.042816
2,6242,0.283627,14.601562,0.019424
3,5025,-0.271659,6.667969,-0.040741
4,1835,0.119991,29.125000,0.004120
5,7987,0.118014,-46.312500,-0.002548
6,3062,0.115411,-35.656250,-0.003237
7,8382,-0.111913,5.585938,-0.020035
8,7926,-0.107046,-15.359375,0.006969
9,4762,0.090722,7.554688,0.012009


**Interpretation:** one neuron's screening score should stand out
sharply from the rest — a strong candidate for the real validation in
Section 6. A large positive score means increasing that neuron's activation
would locally increase the target log-probability; a large negative score
means the opposite.


## 6. Validating the cheap screening with real single-neuron ablation

Screening only estimates a *local, linear* effect. Let's check each of the
top 10 candidates with a real ablation (zeroing that one neuron before
`down_proj`, in an actual forward pass) and compare the real change in
log-probability to what the linear screening predicted.


In [6]:
def ablate_neurons(layer_idx: int, neuron_indices: list[int], model_inputs: dict):
    '''Run a forward pass over model_inputs with the given neurons' activations at layer_idx zeroed out before down_proj.'''
    def pre_hook(module, args, kwargs):
        hidden_states = args[0].clone()
        hidden_states[..., neuron_indices] = 0.0
        return (hidden_states,) + args[1:], kwargs

    handle = model.model.layers[layer_idx].mlp.down_proj.register_forward_pre_hook(pre_hook, with_kwargs=True)
    try:
        with torch.no_grad():
            out = model(**model_inputs)
    finally:
        handle.remove()
    return out.logits[0, -1, :]


validation_rows = []
for neuron_idx, score in zip(top_neurons.tolist(), screening_scores[top_neurons].tolist()):
    ablated_logits = ablate_neurons(LAYER, [neuron_idx], encoded)
    ablated_logprob = torch.log_softmax(ablated_logits.float(), dim=-1)[TARGET_ID].item()
    real_change = ablated_logprob - clean_logprob
    predicted_change = -score  # ablation sets activation to 0, i.e. subtracts it; linear prediction is -gradient*activation
    validation_rows.append({
        "neuron": neuron_idx,
        "predicted_change": predicted_change,
        "real_change": real_change,
        "ablated_logprob": ablated_logprob,
    })

validation_table = pd.DataFrame(validation_rows)
validation_table


,neuron,predicted_change,real_change,ablated_logprob
0,7015,4.449463,1.084582,-0.008286
1,8715,-0.322794,-0.489906,-1.582774
2,6242,-0.283627,-0.304883,-1.397751
3,5025,0.271659,0.258478,-0.834390
4,1835,-0.119991,-0.129495,-1.222363
5,7987,-0.118014,-0.133556,-1.226424
6,3062,-0.115411,-0.122527,-1.215395
7,8382,0.111913,0.109759,-0.983109
8,7926,0.107046,0.104872,-0.987996
9,4762,-0.090722,-0.123757,-1.216625


**A real, honest finding.** For most of these neurons, the real
change tracks the linear prediction reasonably well. But watch the very top
candidate (the one with by far the largest screening score) — while
building this notebook, its real effect was substantially *smaller* than
the linear approximation predicted, even though the direction (sign) was
right. This is notebook 4's "saturation" caution, no longer just a
conceptual warning: a strong effect on an already-confident prediction can
genuinely mean the true relationship has flattened out from where the
local gradient was measured, and only a real ablation reveals that.


In [7]:
best_row = validation_table.reindex(validation_table["real_change"].abs().sort_values(ascending=False).index).iloc[0]
TOP_NEURON = int(best_row["neuron"])

print(f"Most important neuron, confirmed by real ablation: layer {LAYER}, neuron {TOP_NEURON}")
print(f"Clean log P({TARGET_TEXT!r}):                {clean_logprob:.4f}  ({np.exp(clean_logprob)*100:.1f}%)")
print(f"Log P after ablating neuron {TOP_NEURON}:  {best_row['ablated_logprob']:.4f}  ({np.exp(best_row['ablated_logprob'])*100:.1f}%)")


Most important neuron, confirmed by real ablation: layer 27, neuron 7015
Clean log P(' bit'):                -1.0929  (33.5%)
Log P after ablating neuron 7015:  -0.0083  (99.2%)


**Interpretation:** report the real percentages above. If ablating
this single neuron pushes the target token from a modest probability to
something close to certainty, that is a striking, large, and entirely real
causal effect from a single one of 8,960 values in one layer.


## 8. A second, independent method: what does this neuron "write" toward the vocabulary?

Every neuron's activation, when nonzero, adds a fixed direction to the
residual stream: the corresponding column of `down_proj`'s weight matrix.
Because this model's output (unembedding) matrix is *tied* to its input
embedding matrix, we can project that one direction straight through to
vocabulary space — asking "if this neuron fired at full strength, on any
input at all, which tokens would it most push toward or away from?" —
**without running the model on any prompt whatsoever.**

We apply the final layer normalization's learned per-dimension scale
(a fixed weight vector, independent of any specific input) but skip its
activation-dependent magnitude rescaling — that rescaling divides every
token's logit by the same positive number for a given input, so it cannot
change which tokens rank highest, only their absolute scale, which is why
skipping it doesn't affect this ranking.


In [8]:
down_proj_weight = model.model.layers[LAYER].mlp.down_proj.weight.detach().float()
neuron_direction = down_proj_weight[:, TOP_NEURON]
final_norm_weight = model.model.norm.weight.detach().float()
scaled_direction = neuron_direction * final_norm_weight

embedding_matrix = model.get_input_embeddings().weight.detach().float()  # tied with the output/unembedding layer
projected_logits = embedding_matrix @ scaled_direction

top_promoted = torch.topk(projected_logits, 12)
top_suppressed = torch.topk(-projected_logits, 12)

promoted_table = pd.DataFrame({
    "token": [repr(tokenizer.decode([tid])) for tid in top_promoted.indices.tolist()],
    "projected_logit": top_promoted.values.tolist(),
})
suppressed_table = pd.DataFrame({
    "token": [repr(tokenizer.decode([tid])) for tid in top_suppressed.indices.tolist()],
    "projected_logit": (-top_suppressed.values).tolist(),
})

print(f"Neuron {TOP_NEURON}, layer {LAYER} -- tokens this neuron's direction most PROMOTES (prompt-independent):")
display(promoted_table)
print("\nTokens this neuron's direction most SUPPRESSES:")
display(suppressed_table)

target_rank = int((projected_logits > projected_logits[TARGET_ID]).sum().item())
print(f"\n{TARGET_TEXT!r}'s rank among all {len(projected_logits):,} tokens in this neuron's promoted direction: "
      f"{target_rank + 1} (1 = most promoted, {len(projected_logits):,} = most suppressed)")


Neuron 7015, layer 27 -- tokens this neuron's direction most PROMOTES (prompt-independent):


,token,projected_logit
0,'.va',0.275867
1,'鼋',0.274429
2,' jel',0.252904
3,' Smy',0.242471
4,'�',0.236494
5,' startY',0.232789
6,'мен',0.230999
7,'erde',0.229683
8,'elige',0.227046
9,'年第',0.226134



Tokens this neuron's direction most SUPPRESSES:


,token,projected_logit
0,' bit',-1.688149
1,'bit',-1.614204
2,' Bit',-1.589065
3,'Bit',-1.519103
4,' bits',-1.510840
5,'-bit',-1.446197
6,'bits',-1.409379
7,'_bit',-1.392689
8,'Bits',-1.365238
9,'(bit',-1.352320



' bit''s rank among all 151,936 tokens in this neuron's promoted direction: 151936 (1 = most promoted, 151,936 = most suppressed)


**Interpretation — read the real lists above.** If the "promoted"
list looks like unrelated noise but the "suppressed" list is dominated by
spelling and capitalization variants of one specific word — and if that
word ranks at or near dead last among the *entire* vocabulary for this
neuron's direction — that is a genuinely clean, specific finding: this
neuron's job, as far as its output weights alone can tell us, looks like
suppressing that one word's family of tokens, independent of any prompt at
all.


## 9. Why finding this two independent ways matters

Section 6 identified this neuron using a real, causal ablation on **one
specific prompt** — a method that knows nothing about vocabulary meaning,
only about this prompt's actual log-probability. Section 8 identified what
this neuron's direction favors using **only its learned weights** — a
method that knows nothing about our prompt at all, or even that "bit" was
ever a candidate prediction here.

If both independently point to the same specific word, that agreement is
substantially stronger evidence than either method alone — the same
principle notebook 7 demonstrated with attention and ablation, now with an
even sharper contrast, since the logit-lens method here used no prompt
whatsoever.


## 10. What this notebook can — and cannot — establish

We can honestly say, because we computed and validated it directly above:

- one specific neuron, identified independently by real ablation and by a
  prompt-free inspection of its own weights, shows a clean, consistent
  story around one word's token family
- cheap gradient-based screening is a useful way to narrow 8,960
  candidates down to a handful worth checking directly — but Section 6
  showed concretely that its predicted magnitude can be substantially
  wrong for the very neuron that matters most, which is exactly why we
  validated with real ablation rather than trusting the screening score
  alone

We should **not** say:

- that this neuron "means bit" as its sole, complete function — real
  neurons are frequently **polysemantic**: the same neuron can respond to
  several unrelated concepts in different contexts (a well-documented
  phenomenon in interpretability research, often attributed to models
  representing more features than they have neurons for). We only tested
  this one behavioral facet, on one layer, for one target word.
- that the logit-lens projection's exact magnitudes are meaningful — we
  explicitly skipped the RMSNorm's activation-dependent rescaling; the
  *ranking* of tokens is justified by that skip, but the specific numbers
  are not directly comparable to a real forward pass's logits
- that we searched exhaustively — we only examined the final layer, chosen
  because notebooks 5 and 7 both pointed to late layers mattering most for
  this prompt; other layers were not checked
- that this is a demonstrated "circuit" in the full mechanistic-
  interpretability sense — we found one component with a clean story on
  one prompt, not a validated, general algorithm confirmed across many
  inputs


## 11. Key lessons, and the full series so far

| Notebook | Granularity | Kind of evidence |
|---|---|---|
| 3: Attention | Whole layer, averaged over heads | Descriptive, not causal |
| 4: Gradients / occlusion | Whole input, per token position | Local approximation / single-substitution causal test |
| 5: Activation patching | Whole layer's output, per position | Causal intervention, not head- or neuron-resolved |
| 6: Probing | Whole layer's representation | Correlational; tests presence, not use |
| 7: Head ablation | One attention head | Causal intervention, head-resolved |
| 8: Neuron analysis (this notebook) | One MLP neuron | Causal intervention (validated) + prompt-free weight inspection |

1. Some structures (8,960 neurons per layer) are too large to exhaustively
   search the way 12 attention heads were — a cheap screening step,
   validated afterward on real ablations, is a practical alternative.
2. Gradient-based screening's accuracy is not guaranteed at the extreme —
   we found and quantified a real case where it overestimated a genuine
   effect, exactly the "saturation" failure mode notebook 4 named.
3. A neuron's learned output weights can be inspected for what they favor
   in vocabulary space without running the model at all — a genuinely
   different, prompt-independent kind of evidence.
4. Two independently-derived pieces of evidence (causal, prompt-specific;
   descriptive, prompt-free) pointing to the same conclusion is
   meaningfully stronger than either alone.
5. A clean single-neuron story is still a hypothesis about one component,
   one layer, one target — not a proven, general circuit.


## 12. Try your own oilfield sentence

Change the prompt below, then run the cell. It repeats Sections 5-8's
pipeline: screen all 8,960 neurons in the final layer, validate the top
candidates with real ablation, and show the confirmed top neuron's
logit-lens projection.


In [9]:
user_prompt = "The crew ran in hole with the completion assembly and began to"  # <-- change this line

user_encoded = tokenizer(user_prompt, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    user_clean_logits = model(**user_encoded).logits[0, -1, :]
user_target_id = int(torch.argmax(user_clean_logits).item())
user_target_text = tokenizer.decode([user_target_id])
user_clean_logprob = torch.log_softmax(user_clean_logits.float(), dim=-1)[user_target_id].item()
print(f"Target token: {user_target_text!r}")

user_captured = {}
def user_capture_hook(module, inputs, output):
    activation = inputs[0]
    activation.retain_grad()
    user_captured["activation"] = activation

handle = model.model.layers[LAYER].mlp.down_proj.register_forward_hook(user_capture_hook)
user_out = model(**user_encoded)
user_logprob = torch.log_softmax(user_out.logits[0, -1, :].float(), dim=-1)[user_target_id]
user_logprob.backward()
handle.remove()

user_activation = user_captured["activation"][0, -1, :].detach().float()
user_gradient = user_captured["activation"].grad[0, -1, :].float()
user_scores = user_gradient * user_activation
_, user_top_neurons = torch.topk(user_scores.abs(), 10)

user_validation = []
for neuron_idx in user_top_neurons.tolist():
    ablated = ablate_neurons(LAYER, [neuron_idx], user_encoded)
    ablated_lp = torch.log_softmax(ablated.float(), dim=-1)[user_target_id].item()
    user_validation.append({"neuron": neuron_idx, "real_change": ablated_lp - user_clean_logprob})

user_validation_df = pd.DataFrame(user_validation)
user_top_neuron = int(user_validation_df.loc[user_validation_df["real_change"].abs().idxmax(), "neuron"])
print(f"Most important neuron (validated by real ablation): layer {LAYER}, neuron {user_top_neuron}")
display(user_validation_df)

user_direction = model.model.layers[LAYER].mlp.down_proj.weight.detach().float()[:, user_top_neuron] * final_norm_weight
user_projected = embedding_matrix @ user_direction
user_top_promoted = torch.topk(user_projected, 8)
user_top_suppressed = torch.topk(-user_projected, 8)

print("\nThis neuron's direction most PROMOTES:")
for tid, val in zip(user_top_promoted.indices.tolist(), user_top_promoted.values.tolist()):
    print(f"  {tokenizer.decode([tid])!r:20s} {val:.3f}")
print("\nThis neuron's direction most SUPPRESSES:")
for tid, val in zip(user_top_suppressed.indices.tolist(), user_top_suppressed.values.tolist()):
    print(f"  {tokenizer.decode([tid])!r:20s} {-val:.3f}")


Target token: ' install'


Most important neuron (validated by real ablation): layer 27, neuron 3559


,neuron,real_change
0,3559,1.153603
1,8276,-0.164499
2,5641,0.149676
3,2631,-0.162073
4,3240,-0.155787
5,174,0.123323
6,5345,0.124379
7,7987,-0.142541
8,8107,-0.121444
9,137,0.081800



This neuron's direction most PROMOTES:
  ' install'           1.610
  ' installation'      1.600
  '安装'                 1.578
  'install'            1.523
  ' installed'         1.522
  ' Installation'      1.502
  ' Install'           1.496
  ' installing'        1.431

This neuron's direction most SUPPRESSES:
  '�'                  -0.404
  ' �'                 -0.347
  '宗'                  -0.306
  ' Teams'             -0.241
  ' Dzi'               -0.232
  '�'                  -0.232
  ' teams'             -0.229
  'frey'               -0.227


## 13. Optional exercises

1. Test whether the neuron found in Section 7 is specific to "bit" or a
   more general anti-repetition mechanism: try a prompt that repeats a
   *different* word (e.g. "The crew changed the worn packer and installed
   a new" -- does the same neuron suppress "packer," or only "bit"?).
2. In Section 6, change `LAYER` to an earlier layer (e.g. `LAYER = 20`) and
   re-run Sections 5-8. Does a similarly dominant neuron emerge?
3. In Section 8, look at the *second*-most-important neuron from Section 6
   instead of the top one. Does its logit-lens projection tell as clean a
   story?
4. Modify Section 6 to also report each candidate's *rank-correlation*
   between predicted and real changes across all 10 neurons (e.g. using
   `scipy.stats.spearmanr`), as a single summary number for how well
   screening tracked reality this time.
5. Run Section 12 on one of notebook 1's POOH prompts. Is the confirmed top
   neuron in the same position as the one found for the main prompt here?


## 14. Technical appendix

**Model and environment actually used in this run** (printed live, not
hard-coded):


In [10]:
print(f"Model:                {MODEL_NAME}")
print(f"Device:               {DEVICE}")
print(f"Dtype:                {DTYPE}")
print(f"Transformer layers:   {NUM_LAYERS}")
print(f"MLP intermediate size: {INTERMEDIATE_SIZE:,}")
print(f"Tied embeddings:      {model.config.tie_word_embeddings}")
print(f"Python version:       {sys.version.split()[0]}")
print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Random seed:          {SEED}")


Model:                Qwen/Qwen2.5-1.5B-Instruct
Device:               mps
Dtype:                torch.float16
Transformer layers:   28
MLP intermediate size: 8,960
Tied embeddings:      True
Python version:       3.12.13
PyTorch version:      2.14.0
Transformers version: 5.16.1
Random seed:          42


**Why the logit lens needs tied embeddings, restated.** This
technique projects a hidden-space direction straight through the
unembedding matrix. It only makes sense here because this model's
unembedding is literally the same matrix as its input token embeddings
(`tie_word_embeddings: True`, confirmed in Section 2) — for a model with
separate, untied output weights, you would need that model's own distinct
unembedding matrix instead.

**Performance note.** The gradient-based screening in Section 5 needs only
one forward and one backward pass to score all 8,960 neurons in a layer.
Validating the top 10 with real ablation (Section 6) needs 10 more forward
passes. The logit-lens projection (Section 8) needs no forward pass at
all — it's a single matrix-vector product. The whole notebook completes in
a few seconds on this machine's Apple Silicon GPU.
